# Word Embeddings, Vector Representations, POS Tagging, and NER

This practical assumes you already know the basics of text preprocessing from an earlier course.  
Here, the focus is on **representation learning** and **sequence labelling** in NLP.

## In-class path for this lab
Please treat the following as the **core** activities for the live lab:

- inspect the **CoNLL-2003** annotations
- train a compact **Word2Vec-style CBOW** model
- train a compact **GloVe** model from a **co-occurrence matrix**
- compare **CBOW** and **GloVe** using **cosine similarity**, **nearest neighbors**, and **PCA**
- reuse the learned embeddings in a downstream **NER** model

The **POS tagger** and **FastText** appear as **extension/homework** so the notebook remains realistic for the lab session.

## What this notebook covers
- a compact **Word2Vec-style CBOW** model trained from scratch
- a compact **GloVe** model trained from a **word co-occurrence matrix**
- how to inspect and compare the learned embedding spaces using **cosine similarity**, **nearest neighbors**, and **PCA**
- how embeddings can be reused in downstream **NER**
- how **POS tagging** and **FastText** extend the story beyond the core lab

## Exercises in this notebook
There are **3 core exercises** for the live lab. They are designed to help you:
- interpret token-level annotations
- compare **CBOW** and **GloVe**
- connect embedding quality to downstream tagging behavior

Anything marked **Optional extension** can be completed later if time runs out.


## Install dataset library

If needed, uncomment the installation line below.

In [ ]:
#!pip install datasets seqeval scikit-learn

In [ ]:
#pip install transformers datasets==2.18.0 evaluate seqeval

## 1. Core — Setup: load the dataset and label vocabularies

This cell imports the required libraries, loads a compact subset of **CoNLL-2003**, and stores the label names for **POS** and **NER**.

We use a reduced subset so the notebook runs comfortably, while still being large enough to learn meaningful patterns.


In [ ]:
import random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, f1_score, classification_report
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Small enough to run in class, large enough to learn something useful.
N_TRAIN = 1800
N_VALID = 300
N_TEST = 300

dataset = load_dataset("conll2003")
train_raw = dataset["train"].select(range(N_TRAIN))
valid_raw = dataset["validation"].select(range(N_VALID))
test_raw = dataset["test"].select(range(N_TEST))

train_sentences = [ex["tokens"] for ex in train_raw]
valid_sentences = [ex["tokens"] for ex in valid_raw]
test_sentences = [ex["tokens"] for ex in test_raw]

pos_names = dataset["train"].features["pos_tags"].feature.names
ner_names = dataset["train"].features["ner_tags"].feature.names

print(f"Device: {device}")
print(f"Train / Valid / Test sentences: {len(train_raw)} / {len(valid_raw)} / {len(test_raw)}")
print(f"POS tag set size: {len(pos_names)}")
print(f"NER tag set size: {len(ner_names)}")


## 2. Core — Inspect one sentence with aligned POS and NER labels

This cell prints one annotated sentence so you can see how the dataset is organised at the **token level**.

### How to read the POS tags in the sample
The POS tags come from the Penn Treebank tag set. In the example sentence, the most important ones are:

- **NNP**: proper noun, singular (for example, *EU*)
- **VBZ**: verb, 3rd person singular present (for example, *rejects*)
- **JJ**: adjective (for example, *German*, *British*)
- **NN**: common noun, singular or mass (for example, *call*, *lamb*)
- **TO**: the word *to*
- **VB**: base-form verb (for example, *boycott*)
- **.**: punctuation

### How to read the NER tags
The NER labels follow the **IOB scheme**:

- **B-***: beginning of an entity span
- **I-***: inside an entity span
- **O**: outside any named entity

Entity types in CoNLL-2003 include:

- **ORG**: organization
- **PER**: person
- **LOC**: location
- **MISC**: miscellaneous named entities such as nationalities, religious groups, or event-related terms

This first inspection step matters because later we will build models that must predict these token-level labels automatically.


In [ ]:
example_id = 0
example = train_raw[example_id]

sample_df = pd.DataFrame({
    "token": example["tokens"],
    "pos_tag": [pos_names[i] for i in example["pos_tags"]],
    "ner_tag": [ner_names[i] for i in example["ner_tags"]],
})

avg_len = np.mean([len(sent) for sent in train_sentences])

print(f"Average number of tokens per training sentence: {avg_len:.2f}")
print("Example sentence with token-level POS and NER annotations:")
sample_df

## Exercise 1 — Read the annotations

After looking at the sample sentence, answer briefly:

1. Which words in the sentence are easiest to tag with **POS**, and why?
2. Which words are easiest to tag with **NER**, and why?
3. Which task seems to rely more on **local context** rather than only the identity of a single token?


## 3. Core — Build the vocabulary and create CBOW training pairs

This cell builds a vocabulary from the training corpus and converts each sentence into **CBOW context-target pairs**.

Recall the idea behind **CBOW (Continuous Bag-of-Words)**:
- the model sees a local context window
- it predicts the center word
- the embedding matrix is learned through that prediction task


In [ ]:
MIN_FREQ = 2
WINDOW_SIZE = 1
BATCH_SIZE = 1024

token_counter = Counter(tok for sent in train_sentences for tok in sent)
special_tokens = ["<PAD>", "<UNK>"]
vocab = special_tokens + [tok for tok, c in token_counter.items() if c >= MIN_FREQ]

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}
PAD_IDX = word2idx["<PAD>"]
UNK_IDX = word2idx["<UNK>"]

def encode_token(tok):
    return word2idx.get(tok, UNK_IDX)

def generate_cbow_pairs(sentences, window_size=2):
    contexts, targets = [], []
    for sent in sentences:
        if len(sent) < 2 * window_size + 1:
            continue
        ids = [encode_token(tok) for tok in sent]
        for i in range(window_size, len(ids) - window_size):
            context = ids[i - window_size:i] + ids[i + 1:i + window_size + 1]
            target = ids[i]
            contexts.append(context)
            targets.append(target)
    return torch.tensor(contexts, dtype=torch.long), torch.tensor(targets, dtype=torch.long)

cbow_contexts, cbow_targets = generate_cbow_pairs(train_sentences, window_size=WINDOW_SIZE)
cbow_loader = DataLoader(
    TensorDataset(cbow_contexts, cbow_targets),
    batch_size=BATCH_SIZE,
    shuffle=True
)

print(f"Vocabulary size: {len(vocab)}")
print(f"Number of CBOW training pairs: {len(cbow_targets)}")
print("First context-target example:")
print("context ids :", cbow_contexts[0].tolist())
print("context toks:", [idx2word[i] for i in cbow_contexts[0].tolist()])
print("target id   :", int(cbow_targets[0]))
print("target tok  :", idx2word[int(cbow_targets[0])])

## 4. Core — Train a compact Word2Vec-style CBOW model

This cell defines a small **CBOW embedding model**, trains it for a few epochs, and stores both:
- the **initial random embedding matrix**
- the **trained embedding matrix**

That lets us compare random vectors with learned vectors later.

For the lab, we keep the number of epochs deliberately small so training finishes quickly.


In [ ]:
EMBED_DIM = 50
CBOW_EPOCHS = 5
CBOW_LR = 3e-3

class CBOWModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, context_ids):
        emb = self.embedding(context_ids)   # [B, C, D]
        pooled = emb.mean(dim=1)            # [B, D]
        return self.output(pooled)          # [B, V]

cbow_model = CBOWModel(len(vocab), EMBED_DIM).to(device)
initial_embedding_matrix = cbow_model.embedding.weight.detach().cpu().clone()

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(cbow_model.parameters(), lr=CBOW_LR)

for epoch in range(1, CBOW_EPOCHS + 1):
    cbow_model.train()
    total_loss = 0.0
    for context_ids, target_ids in cbow_loader:
        context_ids = context_ids.to(device)
        target_ids = target_ids.to(device)

        optimizer.zero_grad()
        logits = cbow_model(context_ids)
        loss = criterion(logits, target_ids)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * context_ids.size(0)

    avg_loss = total_loss / len(cbow_loader.dataset)
    print(f"CBOW epoch {epoch:02d} | loss = {avg_loss:.4f}")

embedding_matrix = cbow_model.embedding.weight.detach().cpu()

## 5. Core — Build a co-occurrence matrix and train a compact GloVe model

This cell builds **global co-occurrence statistics** from the training corpus and trains a small **GloVe** model.

Conceptually:

- **CBOW / Word2Vec-style training** learns from predicting a center word from its local context
- **GloVe** learns from how often words co-occur across the corpus
- both produce dense vectors, but the learning signals are different

The cell below:
1. builds weighted co-occurrence pairs from the corpus
2. trains a compact GloVe objective
3. stores the final **GloVe embedding matrix** for later comparison

Again, the training budget is intentionally small so this section fits inside the live lab.


In [ ]:
GLOVE_WINDOW_SIZE = WINDOW_SIZE
GLOVE_EPOCHS = 6
GLOVE_LR = 1e-2
GLOVE_BATCH_SIZE = 8192
X_MAX = 100.0
ALPHA = 0.75

cooccurrence = Counter()
for sent in train_sentences:
    ids = [encode_token(tok) for tok in sent]
    for center_pos, center_id in enumerate(ids):
        left = max(0, center_pos - GLOVE_WINDOW_SIZE)
        right = min(len(ids), center_pos + GLOVE_WINDOW_SIZE + 1)
        for ctx_pos in range(left, right):
            if ctx_pos == center_pos:
                continue
            ctx_id = ids[ctx_pos]
            distance = abs(center_pos - ctx_pos)
            cooccurrence[(center_id, ctx_id)] += 1.0 / distance

cooc_i = torch.tensor([pair[0] for pair in cooccurrence.keys()], dtype=torch.long)
cooc_j = torch.tensor([pair[1] for pair in cooccurrence.keys()], dtype=torch.long)
cooc_x = torch.tensor(list(cooccurrence.values()), dtype=torch.float32)

class GloVeModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.word_emb = nn.Embedding(vocab_size, embed_dim)
        self.context_emb = nn.Embedding(vocab_size, embed_dim)
        self.word_bias = nn.Embedding(vocab_size, 1)
        self.context_bias = nn.Embedding(vocab_size, 1)

    def forward(self, word_ids, context_ids):
        w = self.word_emb(word_ids)
        c = self.context_emb(context_ids)
        b_w = self.word_bias(word_ids).squeeze(-1)
        b_c = self.context_bias(context_ids).squeeze(-1)
        return (w * c).sum(dim=1) + b_w + b_c

def glove_weighting(x, x_max=X_MAX, alpha=ALPHA):
    return torch.where(x < x_max, (x / x_max) ** alpha, torch.ones_like(x))

glove_model = GloVeModel(len(vocab), EMBED_DIM).to(device)
glove_optimizer = torch.optim.Adam(glove_model.parameters(), lr=GLOVE_LR)

for epoch in range(1, GLOVE_EPOCHS + 1):
    perm = torch.randperm(len(cooc_x))
    epoch_loss = 0.0

    for start in range(0, len(perm), GLOVE_BATCH_SIZE):
        batch_idx = perm[start:start + GLOVE_BATCH_SIZE]
        word_ids = cooc_i[batch_idx].to(device)
        context_ids = cooc_j[batch_idx].to(device)
        counts = cooc_x[batch_idx].to(device)

        weights = glove_weighting(counts)
        targets = torch.log(counts)

        glove_optimizer.zero_grad()
        preds = glove_model(word_ids, context_ids)
        loss = (weights * (preds - targets) ** 2).mean()
        loss.backward()
        glove_optimizer.step()

        epoch_loss += loss.item() * len(batch_idx)

    avg_loss = epoch_loss / len(cooc_x)
    print(f"GloVe epoch {epoch:02d} | weighted MSE = {avg_loss:.4f}")

glove_embedding_matrix = (
    glove_model.word_emb.weight.detach().cpu() +
    glove_model.context_emb.weight.detach().cpu()
) / 2.0

print(f"Number of non-zero co-occurrence pairs: {len(cooc_x)}")
print("Example co-occurrence counts:")
for k, ((i_id, j_id), value) in enumerate(cooccurrence.items()):
    print(f"  ({idx2word[i_id]}, {idx2word[j_id]}) -> {value:.2f}")
    if k == 4:
        break


## 6. Core — Inspect and compare the CBOW and GloVe embedding spaces

This section asks two standard embedding questions for **both** models:

1. Are intuitively related words assigned similar vectors?
2. What words sit closest to a given query word in the learned vector space?

It also compares one similarity score before and after **CBOW** training so you can see that the geometry is learned rather than fixed.


In [ ]:
def resolve_token(token):
    for cand in (token, token.lower(), token.capitalize(), token.upper()):
        if cand in word2idx:
            return cand
    return None

def cosine_similarity(word_a, word_b, matrix):
    tok_a = resolve_token(word_a)
    tok_b = resolve_token(word_b)
    if tok_a is None or tok_b is None:
        return None
    vec_a = matrix[word2idx[tok_a]]
    vec_b = matrix[word2idx[tok_b]]
    return torch.nn.functional.cosine_similarity(vec_a, vec_b, dim=0).item()

def nearest_neighbors(query_word, matrix, k=5):
    tok = resolve_token(query_word)
    if tok is None:
        return []
    query_vec = matrix[word2idx[tok]]
    sims = torch.nn.functional.cosine_similarity(query_vec.unsqueeze(0), matrix, dim=1)
    best_ids = torch.topk(sims, k + 1).indices.tolist()
    neighbors = []
    for idx in best_ids:
        candidate = idx2word[idx]
        if candidate != tok:
            neighbors.append((candidate, float(sims[idx])))
        if len(neighbors) == k:
            break
    return neighbors

embedding_spaces = {
    "CBOW": embedding_matrix,
    "GloVe": glove_embedding_matrix,
}

pairs = [
    ("London", "Paris"),
    ("Germany", "France"),
    ("company", "bank"),
    ("football", "baseball"),
]

for name, matrix in embedding_spaces.items():
    print(f"\n{name} cosine similarities")
    print("-" * 40)
    for a, b in pairs:
        score = cosine_similarity(a, b, matrix)
        print(f"{a:>10} vs {b:<10} -> {score}")

for query in ["London", "Germany", "President"]:
    print(f"\nQuery word: {resolve_token(query)}")
    for name, matrix in embedding_spaces.items():
        print(f"  {name} nearest neighbors:")
        for word, score in nearest_neighbors(query, matrix=matrix, k=5):
            print(f"    {word:<15} {score:.4f}")

print("\nRandom-vs-trained CBOW quick check for 'London' and 'Paris':")
print("Initial :", cosine_similarity("London", "Paris", initial_embedding_matrix))
print("CBOW    :", cosine_similarity("London", "Paris", embedding_matrix))
print("GloVe   :", cosine_similarity("London", "Paris", glove_embedding_matrix))


## Exercise 2 — Compare CBOW and GloVe

Use the output of the analysis cell to answer:

1. For the word pairs shown, which model gives the more intuitive similarities?
2. For the nearest-neighbor lists, where does **CBOW** look stronger?
3. Where does **GloVe** look stronger?

Optional extension:
- change `WINDOW_SIZE`
- change `EMBED_DIM`

Then compare how the two embedding spaces change.


## 7. Core — Visualize selected embeddings with PCA for CBOW and GloVe

This cell projects a small set of learned word vectors from the full embedding space down to **2D** using **PCA**.

We plot **CBOW** and **GloVe** side by side so you can compare how the two objectives organize the same vocabulary.

Remember: PCA is useful for inspection, but a 2D projection can only show part of the structure of a higher-dimensional space.


In [ ]:
candidate_words = [
    "London", "Paris", "Berlin", "Germany", "France", "Italy",
    "President", "minister", "company", "bank", "market",
    "football", "baseball", "tennis"
]

plot_words = [w for w in candidate_words if resolve_token(w) is not None]
plot_words = [resolve_token(w) for w in plot_words]
plot_words = list(dict.fromkeys(plot_words))  # unique while preserving order

cbow_vecs = np.stack([embedding_matrix[word2idx[w]].numpy() for w in plot_words])
glove_vecs = np.stack([glove_embedding_matrix[word2idx[w]].numpy() for w in plot_words])

cbow_coords = PCA(n_components=2, random_state=SEED).fit_transform(cbow_vecs)
glove_coords = PCA(n_components=2, random_state=SEED).fit_transform(glove_vecs)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, coords, title in zip(
    axes,
    [cbow_coords, glove_coords],
    ["CBOW embedding space", "GloVe embedding space"],
):
    ax.scatter(coords[:, 0], coords[:, 1], s=60)
    for i, word in enumerate(plot_words):
        ax.text(coords[i, 0] + 0.02, coords[i, 1] + 0.02, word, fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Exercise 3 — Read the CBOW vs GloVe plots

Look at the PCA plots and answer:

1. Do cities, countries, or sports words form any visible clusters in **either** model?
2. Which model produces the more interpretable 2D layout for this small corpus?
3. Why should we be careful not to over-interpret a **2D projection** of a higher-dimensional space?


## 8. Core — Encode the sequence-labelling datasets for downstream tagging

This cell converts the token sequences into integer IDs and creates padded mini-batches for the downstream tagging tasks.

We keep the same vocabulary as above so the sequence taggers can optionally start from:
- **random embeddings**
- the learned **CBOW embeddings**
- the learned **GloVe embeddings**

In the live lab, we will use these utilities for **NER**.  
The **POS tagger** is kept as an optional extension.


In [ ]:
def encode_sequence_dataset(raw_dataset, label_field):
    encoded = []
    for ex in raw_dataset:
        encoded.append({
            "tokens": ex["tokens"],
            "input_ids": [encode_token(tok) for tok in ex["tokens"]],
            "labels": ex[label_field],
        })
    return encoded

def make_collate_fn(pad_idx):
    def collate_fn(batch):
        max_len = max(len(item["input_ids"]) for item in batch)
        input_ids, labels, tokens = [], [], []
        for item in batch:
            seq_len = len(item["input_ids"])
            pad_len = max_len - seq_len
            input_ids.append(item["input_ids"] + [pad_idx] * pad_len)
            labels.append(item["labels"] + [-100] * pad_len)
            tokens.append(item["tokens"])
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
            "tokens": tokens,
        }
    return collate_fn

collate_fn = make_collate_fn(PAD_IDX)

def make_sequence_loaders(label_field, batch_size=64):
    train_ds = encode_sequence_dataset(train_raw, label_field)
    valid_ds = encode_sequence_dataset(valid_raw, label_field)
    test_ds = encode_sequence_dataset(test_raw, label_field)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
    return train_loader, valid_loader, test_loader

pos_train_loader, pos_valid_loader, pos_test_loader = make_sequence_loaders("pos_tags")
ner_train_loader, ner_valid_loader, ner_test_loader = make_sequence_loaders("ner_tags")

batch = next(iter(pos_train_loader))
print("Sequence batch shapes:")
print("input_ids:", tuple(batch["input_ids"].shape))
print("labels   :", tuple(batch["labels"].shape))

## 9. Core — Define a BiLSTM tagger and shared training/evaluation utilities

This cell defines a compact **BiLSTM sequence tagger** and the shared helpers used for downstream **NER** and the optional **POS** extension.

The evaluation helpers include:
- token-level **accuracy**
- token-level **macro-F1**
- a **classification report** so you can see per-label behavior


In [ ]:
class BiLSTMTagger(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim,
        hidden_dim,
        num_labels,
        pad_idx,
        pretrained_weights=None,
        freeze_embeddings=False,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if pretrained_weights is not None:
            self.embedding.weight.data.copy_(pretrained_weights)
        self.embedding.weight.requires_grad = not freeze_embeddings

        self.encoder = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim // 2,
            batch_first=True,
            bidirectional=True,
        )
        self.classifier = nn.Linear(hidden_dim, num_labels)

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x, _ = self.encoder(x)
        return self.classifier(x)

def get_pretrained_init(init_source):
    if init_source == "random":
        return None
    if init_source == "cbow":
        return embedding_matrix
    if init_source == "glove":
        return glove_embedding_matrix
    raise ValueError("init_source must be one of: 'random', 'cbow', 'glove'")

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * input_ids.size(0)
    return total_loss / len(loader.dataset)

@torch.no_grad()
def collect_predictions(model, loader):
    model.eval()
    y_true, y_pred = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids)
        preds = logits.argmax(dim=-1)
        mask = labels != -100

        y_true.extend(labels[mask].cpu().tolist())
        y_pred.extend(preds[mask].cpu().tolist())

    return y_true, y_pred

@torch.no_grad()
def evaluate_tagger(model, loader, criterion):
    model.eval()
    total_loss = 0.0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)

        logits = model(input_ids)
        loss = criterion(logits.view(-1, logits.size(-1)), labels.view(-1))
        total_loss += loss.item() * input_ids.size(0)

    y_true, y_pred = collect_predictions(model, loader)
    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }

@torch.no_grad()
def classification_report_for_loader(model, loader, label_names):
    y_true, y_pred = collect_predictions(model, loader)
    return classification_report(
        y_true,
        y_pred,
        labels=list(range(len(label_names))),
        target_names=label_names,
        zero_division=0,
    )

@torch.no_grad()
def predict_tags(model, tokens, label_names):
    model.eval()
    input_ids = torch.tensor([[encode_token(tok) for tok in tokens]], dtype=torch.long).to(device)
    preds = model(input_ids).argmax(dim=-1).squeeze(0).cpu().tolist()
    return list(zip(tokens, [label_names[i] for i in preds[:len(tokens)]]))


## 10. Core — Train and evaluate an NER tagger with pretrained embedding initialization

This is the **main downstream task for the live lab**.

You can choose the initialization source with:

- `INIT_SOURCE = "random"`
- `INIT_SOURCE = "cbow"`
- `INIT_SOURCE = "glove"`

Because NER is harder and more label-imbalanced than POS, the **classification report** is especially useful here.


In [ ]:
INIT_SOURCE = "cbow"   # choose from: "random", "cbow", "glove"
TAGGER_EPOCHS = 4
TAGGER_LR = 2e-3
HIDDEN_DIM = 96

ner_model = BiLSTMTagger(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_labels=len(ner_names),
    pad_idx=PAD_IDX,
    pretrained_weights=get_pretrained_init(INIT_SOURCE),
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(ner_model.parameters(), lr=TAGGER_LR)

for epoch in range(1, TAGGER_EPOCHS + 1):
    train_loss = train_one_epoch(ner_model, ner_train_loader, optimizer, criterion)
    valid_metrics = evaluate_tagger(ner_model, ner_valid_loader, criterion)
    print(
        f"NER epoch {epoch:02d} | train loss = {train_loss:.4f} | "
        f"valid acc = {valid_metrics['accuracy']:.4f} | "
        f"valid macro-F1 = {valid_metrics['macro_f1']:.4f}"
    )

ner_test_metrics = evaluate_tagger(ner_model, ner_test_loader, criterion)
print(f"\nNER test metrics ({INIT_SOURCE} init):", ner_test_metrics)
print("\nNER classification report:")
print(classification_report_for_loader(ner_model, ner_test_loader, ner_names))

sample_tokens = ["Barack", "Obama", "visited", "Berlin", "."]
print("\nNER predictions on a short example:")
print(predict_tags(ner_model, sample_tokens, ner_names))


## 11. Optional extension — Train and evaluate a POS tagger

If you finish the core lab early, you can run this section to compare a second sequence-labelling task.

As above, choose the initialization source with:

- `INIT_SOURCE = "random"`
- `INIT_SOURCE = "cbow"`
- `INIT_SOURCE = "glove"`

POS tagging is often easier than NER, so it is a useful contrastive extension.


In [ ]:
INIT_SOURCE = "cbow"   # choose from: "random", "cbow", "glove"
TAGGER_EPOCHS = 3
TAGGER_LR = 2e-3
HIDDEN_DIM = 96

pos_model = BiLSTMTagger(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_labels=len(pos_names),
    pad_idx=PAD_IDX,
    pretrained_weights=get_pretrained_init(INIT_SOURCE),
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-100)
optimizer = torch.optim.Adam(pos_model.parameters(), lr=TAGGER_LR)

for epoch in range(1, TAGGER_EPOCHS + 1):
    train_loss = train_one_epoch(pos_model, pos_train_loader, optimizer, criterion)
    valid_metrics = evaluate_tagger(pos_model, pos_valid_loader, criterion)
    print(
        f"POS epoch {epoch:02d} | train loss = {train_loss:.4f} | "
        f"valid acc = {valid_metrics['accuracy']:.4f} | "
        f"valid macro-F1 = {valid_metrics['macro_f1']:.4f}"
    )

pos_test_metrics = evaluate_tagger(pos_model, pos_test_loader, criterion)
print(f"\nPOS test metrics ({INIT_SOURCE} init):", pos_test_metrics)
print("\nPOS classification report:")
print(classification_report_for_loader(pos_model, pos_test_loader, pos_names))

sample_tokens = test_raw[0]["tokens"]
print("\nPOS predictions on a sample sentence:")
print(predict_tags(pos_model, sample_tokens, pos_names)[:12])


## Optional extension and follow-up assignment

### Optional extension — Compare initialization sources on NER
In the NER cell above, the sequence tagger can start from **random**, **CBOW**, or **GloVe** embeddings.

If time permits:

1. Set `INIT_SOURCE = "random"` and rerun the NER cell.
2. Set `INIT_SOURCE = "cbow"` and rerun it.
3. Set `INIT_SOURCE = "glove"` and rerun it.

Then compare the validation/test metrics and the classification report.

Reflect on:
- whether pretrained initialization helps in this compact setting
- whether **local-context CBOW** or **global co-occurrence GloVe** gives the better starting point for NER

### Follow-up homework — FastText
FastText extends word embeddings by representing each word through **character n-grams** as well as the whole word.

For a short follow-up assignment:

1. Read how FastText builds a word vector from subword units.
2. Explain why FastText is often better for **rare words**, **morphologically rich words**, and **out-of-vocabulary words**.
3. Pick **3 tokens** from this dataset that you think FastText would represent better than plain CBOW or GloVe, and justify your choices.


## Wrap-up

This lab was streamlined so that the **core in-class path** fits a 75-minute session:

- **CBOW** learned embeddings from local context windows by predicting a center word from nearby words.
- **GloVe** learned embeddings from global co-occurrence counts stored in a co-occurrence matrix.
- Similarity, nearest neighbors, and PCA let us inspect and compare the two vector spaces directly.
- **NER** showed how learned embeddings can be reused in a downstream sequence-labelling model.
- **POS tagging** and **FastText** remain natural extensions beyond the live session.

If you revisit this notebook later, the most useful next steps are:
1. compare `INIT_SOURCE = "random"`, `"cbow"`, and `"glove"` more systematically
2. run the optional POS section
3. complete the FastText homework prompt
